# Session 6 — Information Geometry

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

In S5 we put **distributions** in our hands. In S6 we ask a deeper question: what does the
*space* of all distributions look like? The answer is a **curved geometry** — the
**information geometry** of Fisher–Rao — and, crucially, it is *not the only one*. The
same simplex carries a second geometry, that of **optimal transport**, and the two answer
different questions. Holding both in mind is the bridge we will cross all the way to QOT.

## 0. What you'll be able to do

- Place a categorical distribution on the **2-simplex** and see why it is a curved space.
- Compute the **Fisher information metric** for the Bernoulli family and see it diverge
  at the corners.
- Trace a **Fisher–Rao geodesic** between two distributions on the simplex and compare it
  to the linear (mixture) path.
- Compare the **information** interpolation (mixture) with the **transport** interpolation
  (Wasserstein) on the same pair of bumps — the punchline of the session.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.geometry.info_geometry import (
    bernoulli_fisher,
    fisher_rao_distance,
    fisher_rao_geodesic,
    mixture_interpolation,
    wasserstein_interpolation_1d,
)

np.random.seed(42)
viz.use_course_style()

## 1. The probability simplex

A categorical distribution over $K$ outcomes is a vector $p \in \mathbb{R}^K$ with
$p_i \ge 0$ and $\sum_i p_i = 1$. For $K=3$ this is a *triangle* — the **2-simplex**. The
three **corners** $\delta_1, \delta_2, \delta_3$ are the *certain* outcomes
(all mass on one symbol); the **center** $(1/3, 1/3, 1/3)$ is the *uniform* distribution;
the **edges** are the "two-outcome only" slices (one of the three masses is zero).

In [ ]:
examples = {
    "uniform $(1/3, 1/3, 1/3)$": np.array([1/3, 1/3, 1/3]),
    "skewed $(0.7, 0.2, 0.1)$": np.array([0.7, 0.2, 0.1]),
    "almost $\\delta_1$": np.array([0.92, 0.04, 0.04]),
    "on an edge $(0.5, 0.5, 0)$": np.array([0.5, 0.5, 0.0]),
}
viz.plot_simplex_points(examples)
plt.show()

**Read the figure.** Each point inside the triangle *is* a distribution. The vertices are
the deterministic outcomes (one symbol with probability 1), the center is the uniform
distribution, the edges are the boundary where one outcome has been forbidden. Our goal
for the rest of the session is to ask: **what is the natural way to measure distance in
this triangle?** Spoiler — there is more than one answer.

## 2. The Fisher information metric

Two nearby distributions $p$ and $p + \mathrm{d}p$ are "close" in **statistical** sense if
samples drawn from $p$ cannot easily distinguish them from samples drawn from $p+\mathrm{d}p$.
The natural metric for that is the **Fisher information metric**:

$$ g_{ij}(\theta) = \mathbb{E}_{p_\theta}\!\left[\frac{\partial \log p_\theta}{\partial \theta_i}
\,\frac{\partial \log p_\theta}{\partial \theta_j}\right]. $$

For the **Bernoulli** family $p(x;\theta) = (1-\theta,\theta)$, a direct calculation gives

$$ I(\theta) = \frac{1}{\theta(1-\theta)}. $$

Notice — the metric **blows up** near $\theta = 0$ and $\theta = 1$. The boundary of the
simplex is *infinitely far away* in the Fisher–Rao geometry. The simplex is **not flat**.

In [ ]:
thetas = np.linspace(0.005, 0.995, 400)
fisher = np.array([bernoulli_fisher(t) for t in thetas])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(thetas, fisher, color=viz.SOURCE_COLOR, lw=2)
ax.axvline(0.5, color=viz.TARGET_COLOR, linestyle="--", label=r"$\theta = 1/2$ (flattest)")
ax.set_yscale("log")
ax.set_xlabel(r"Bernoulli parameter  $\theta$")
ax.set_ylabel(r"Fisher information  $I(\theta) = 1 / [\theta(1-\theta)]$   (log scale)")
ax.set_title("Fisher information blows up at the corners of the simplex", pad=12)
ax.legend()
plt.show()

**Read the figure.** Minimum $I(1/2) = 4$ at the *flattest* point of the simplex; the
metric **diverges** as $\theta \to 0$ or $\theta \to 1$ (log-scale vertical axis). Read it
geometrically: moving from $\theta = 0.5$ to $\theta = 0.51$ is "easy" (the metric is small
there); moving from $\theta = 0.99$ to $\theta = 0.999$ — which looks like a tiny step in
Euclidean coordinates — is *huge* in Fisher–Rao distance. The simplex *stretches* near
its boundary.

## 3. Fisher–Rao geodesics on the simplex

The remarkable thing is that the Fisher–Rao geometry on the simplex has an *explicit*
description: the change of variable $\phi_i = \sqrt{p_i}$ maps the simplex to a piece of
the unit sphere $\{\phi : \sum \phi_i^2 = 1\}$, and the Fisher–Rao metric becomes the
**round sphere metric**. The geodesic distance is the **angle** between sqrt-vectors
(Rao, 1945; Bhattacharyya, 1943):

$$ d_{\mathrm{FR}}(p, q) = 2\,\arccos\!\left(\sum_i \sqrt{p_i\, q_i}\right) \in [0, \pi]. $$

Geodesics are then **great-circle arcs** (`slerp`) on the sphere of $\sqrt{p}$, projected
back by squaring. Let's check the distance on a few pairs.

In [ ]:
uniform = np.array([1/3, 1/3, 1/3])
print(f"d_FR(uniform, [0.7, 0.2, 0.1]) = {fisher_rao_distance(uniform, [0.7, 0.2, 0.1]):.3f}")
print(f"d_FR([1, 0, 0], [0, 1, 0])     = {fisher_rao_distance([1, 0, 0], [0, 1, 0]):.3f}   (= π, disjoint support)")
print(f"d_FR(p, p)                     = {fisher_rao_distance([0.5, 0.3, 0.2], [0.5, 0.3, 0.2]):.3f}")

**Read the output.** Identical distributions are at distance $0$; distributions with
**disjoint supports** are at the *maximum* possible distance $\pi$ (sqrt-vectors orthogonal).
Everything else is in between. Now let's *draw* a geodesic on the simplex.

In [ ]:
# Two endpoints near opposite edges, plus a third for a second arc.
p0 = np.array([0.10, 0.85, 0.05])
p1 = np.array([0.85, 0.10, 0.05])
p2 = np.array([0.05, 0.05, 0.90])

ts = np.linspace(0.0, 1.0, 80)
fr_arc      = np.array([fisher_rao_geodesic(p0, p1, t) for t in ts])
linear_path = np.array([mixture_interpolation(p0, p1, t) for t in ts])
fr_arc_to_top = np.array([fisher_rao_geodesic(p0, p2, t) for t in ts])

viz.plot_simplex_paths(
    {
        "Fisher–Rao geodesic ($p_0 \\to p_1$)": fr_arc,
        "linear / mixture path ($p_0 \\to p_1$)": linear_path,
        "Fisher–Rao geodesic ($p_0 \\to p_2$)": fr_arc_to_top,
    },
    endpoints={"$p_0$": p0, "$p_1$": p1, "$p_2$": p2},
    title="Fisher–Rao geodesics vs the linear (mixture) path on the 2-simplex",
)
plt.show()

**Read the figure.** Compare the **green Fisher–Rao arc** to the **violet straight line**
between $p_0$ and $p_1$: the FR geodesic **bows away from the boundary** (here, away from
the $\delta_1$–$\delta_2$ edge). That bow is exactly the curvature we read off the
Bernoulli plot — the metric stretches near the edge, so the shortest path prefers to cut
through the interior. The third arc ($p_0 \to p_2$) shows the same bowing pattern from a
different direction. The mixture path treats the simplex as **flat**; the Fisher–Rao
geodesic treats it as **curved**.

## 4. A second geometry on the same simplex: optimal transport

So far, every distance we used — KL, mutual information, Fisher–Rao — was a *"vertical"*
metric: it compared masses bin by bin, **ignoring** what the bins are. That is the right
view if the bins are unrelated symbols (heads/tails, words in a vocabulary).

But often the bins **have their own geometry** — positions on a line, pixels on an image,
neurons on a cortex — and *moving* mass between them has a cost. That is the
**transport geometry** (Wasserstein) we will spend movements M3 and M4 on. It gives a
*different* distance, a *different* geodesic, and a *different* answer to "what is the
natural path between two distributions".

To see the contrast, take two Gaussian bumps on a 1-D line, well-separated. We will
interpolate them in **two** geometries.

In [ ]:
support = np.linspace(0.0, 24.0, 200)
def bump(center, sigma=1.2):
    raw = np.exp(-0.5 * ((support - center) / sigma) ** 2)
    return raw / raw.sum()

p_left  = bump(center=4.0)
p_right = bump(center=20.0)

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(support, p_left,  color=viz.SOURCE_COLOR, lw=2, label="$p_0$ centered at $x=4$")
ax.plot(support, p_right, color=viz.TARGET_COLOR, lw=2, label="$p_1$ centered at $x=20$")
ax.set_xlabel("position  $x$")
ax.set_ylabel("probability mass")
ax.set_title("Two distributions on a line — same simplex, two metrics ahead", pad=12)
ax.legend()
plt.show()

**Read the figure.** Two well-separated bumps with the same width. We will now ask: what
is the "halfway distribution"? Two different geometries will give two different answers.

## 5. Two geometries, two interpolations — the punchline

For $t \in \{0, 0.25, 0.5, 0.75, 1\}$ we compute:

- the **mixture interpolation** (the *information* / mass-axis answer):
  $\;p_t(x) = (1-t)\,p_0(x) + t\,p_1(x)$ — a linear combination of masses bin by bin;
- the **Wasserstein interpolation** (the *transport* / position-axis answer): each unit of
  mass at quantile $u$ in $p_0$ slides linearly to the quantile-$u$ position in $p_1$
  (McCann, 1997).

These are the two natural "halfway distributions". Watch what they do.

In [ ]:
ts = [0.0, 0.25, 0.5, 0.75, 1.0]
mix_snaps = [mixture_interpolation(p_left, p_right, t) for t in ts]
w2_snaps  = [wasserstein_interpolation_1d(p_left, p_right, support, t) for t in ts]

viz.plot_interpolation_panel(
    support,
    ts,
    {
        "mix": mix_snaps,
        "w2":  w2_snaps,
    },
    titles={
        "mix": "Mixture interpolation — amplitudes morph (bimodal at $t=0.5$)",
        "w2":  "Wasserstein interpolation — the peak *slides* (single mode)",
    },
)
plt.show()

**Read both panels — this is the whole session.**

- **Top (mixture / information).** At $t = 0.5$ the distribution is **bimodal**: half the
  mass at $x = 4$, half at $x = 20$, and **nothing in between**. The mixture does not
  care that there is empty space between the two peaks; it interpolates *amplitudes*
  bin by bin. The "halfway distribution" is a *mixture*, not a *displacement*.
- **Bottom (Wasserstein / transport).** The peak **slides** smoothly from $x = 4$ to
  $x = 20$, staying a single bump of (almost) the same width. Wasserstein **knows**
  that the bins live on a line and that empty space between two piles costs something
  to traverse.

Same starting and ending distributions; same simplex. **Two geometries, two different
"shortest paths".** The Fisher–Rao bow we saw in §3 is the curved cousin of the mixture
path — both belong to the **information** family. The Wasserstein interpolation belongs
to a **different family altogether**, and that is the entire reason we will spend the
next nine sessions on optimal transport, and then on its quantum lift.

*(Cf. Peyré & Cuturi, *Computational Optimal Transport*, Fig. 7.4.)*

## 6. Dictionary update

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ |
| Shannon entropy $H(p)$ | von Neumann entropy $S(\rho)$ |
| KL divergence $D(p\|q)$ | Umegaki relative entropy $S(\rho\|\sigma)$ *(S7)* |
| **Fisher–Rao metric on the simplex** | **Bures / BKM metric on density matrices** *(S7)* |
| **mixture (m-) vs exponential (e-) connections** | **quantum dual connections** *(S7)* |
| Wasserstein geometry on probability distributions | quantum Wasserstein on density matrices *(S13–S14)* |

## 7. Your turn

1. **Corners are far.** Pick a trinomial $p \approx (0.99, 0.005, 0.005)$ and a tiny
   perturbation $q \approx (0.99, 0.01, 0.0)$. Compute $d_{\mathrm{FR}}(p, q)$ and
   the Euclidean distance $\|p - q\|_2$. Why is the Fisher–Rao distance so much larger?
2. **Bimodality threshold.** In §5 we used bumps separated by $\Delta x = 16$ with width
   $\sigma = 1.2$. Move the right bump in toward the left one and find the separation at
   which the mixture midpoint stops being bimodal. Compare to $2\sigma$.
3. **Triangle of geodesics.** Trace the three Fisher–Rao geodesics
   $p_0 \!\to\! p_1$, $p_1 \!\to\! p_2$, $p_2 \!\to\! p_0$ using the three points of §3 and
   plot them together. Do they bound a region with larger or smaller area than the
   Euclidean triangle through the same vertices?

## Conclusions & references

- The simplex is a **curved space** under the Fisher–Rao metric — corners are infinitely
  far away.
- **Geodesics**: great-circle arcs on the sphere of $\sqrt{p}$ (Rao–Bhattacharyya).
- The simplex carries **a second geometry** — Wasserstein — that *sees* the ground space
  and **slides** mass instead of *morphing* amplitudes.
- The whole rest of the course is the dialogue between these two geometries: M3 develops
  Wasserstein, S10 reveals that **Sinkhorn = KL projection onto a transport polytope**
  (the bridge), M4 quantizes everything.
- **Next (S7 — Quantum information theory):** lift entropy, KL, mutual information, and
  the Fisher metric to **density matrices**. The dictionary's information row fills in.

**References:** S.-i. Amari, *Information Geometry and Its Applications* (Springer, 2016);
F. Nielsen, "An elementary introduction to information geometry", arXiv:1808.08271 (2020);
G. Peyré & M. Cuturi, *Computational Optimal Transport* (NoW, 2019), ch. 7;
R. McCann, "A convexity principle for interacting gases", Adv. Math. 128, 153 (1997).
Previous: `notebooks/s05_information.ipynb`.